# The deeperfly pipeline, step by step

This notebook reproduces what **`deeperfly run`** does, but one module at a time,
so every intermediate output is visible and plotted. We run on the synchronized
7-camera recording in `data/images/` (one JPEG per camera per frame,
`camera_<cam>_img_<frame>.jpg`).

The `run` CLI (`src/deeperfly/cli.py::_cmd_run`) wires together:

1. **`cameras.CameraGroup`** + **`skeleton.Skeleton`** &mdash; the camera rig and what we track.
2. **`pose2d` detector** &mdash; a stacked-hourglass network producing per-joint heatmaps.
3. **`pose2d.inference`** &mdash; preprocess &rarr; heatmaps &rarr; 2D skeletons per camera.
4. **`pipeline.run_from_points2d`** &mdash; visibility masking &rarr; camera calibration
   (bundle adjustment) &rarr; triangulation with reprojection-outlier rejection &rarr;
   temporal smoothing &rarr; a saved **`io.PoseResult`**.

We unfold steps 1&ndash;4 below and visualize each stage with matplotlib. The final cell
calls `run_from_points2d` directly to show it is exactly the same thing in one call.

## Setup

Import the precise pieces the CLI uses from each module, and point at the data.
`N_FRAMES` controls how much of the 900-frame recording we process &mdash; small
enough to run quickly, large enough for calibration to have something to chew on.

In [ ]:
import tomllib
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt
import imageio.v3 as iio

# The exact modules `deeperfly run` composes:
from deeperfly.cameras import CameraGroup
from deeperfly.skeleton import Skeleton
from deeperfly.pose2d import backends, inference
from deeperfly.pose2d.download import jax_weights_path, torch_weights_path
from deeperfly.triangulate import apply_visibility, triangulate, reprojection_error
from deeperfly.pipeline import calibrate, reconstruct, run_from_points2d
from deeperfly.correction import smooth_one_euro
from deeperfly.io import PoseResult
from deeperfly import viz

# Render plots inline. (Set this AFTER importing `viz`, which defaults matplotlib
# to the headless "Agg" backend on import -- the magic re-asserts the inline one.)
%matplotlib inline

# Locate the repo root (so the notebook works regardless of the launch dir).
REPO = Path.cwd()
while not (REPO / "pyproject.toml").exists() and REPO != REPO.parent:
    REPO = REPO.parent
IMAGES_DIR = REPO / "data" / "images"
CONFIG = REPO / "examples" / "cameras.toml"

# Camera indices 0..6 in the filenames map onto the canonical fly-rig names,
# in the order the example config lists them (right-hind .. left-hind).
CAMERA_NAMES = ["rh", "rm", "rf", "f", "lf", "lm", "lh"]
N_FRAMES = 60  # frames to process (full recording is 900)
BACKEND = "jax"  # detector backend: "jax" (default) or "torch"
FPS = 100.0

n_avail = len(list(IMAGES_DIR.glob("camera_0_img_*.jpg")))
print(f"repo:    {REPO}")
print(f"images:  {IMAGES_DIR}  ({n_avail} frames/camera available)")
print(f"cameras: {CAMERA_NAMES}")

## Step 1 &mdash; the camera rig and the skeleton

`Skeleton.fly()` is the rig-independent description of *what* is tracked: 38
points (right `0..18`, left `19..37`), grouped into limbs, joined by bones, with a
per-camera visibility table. `CameraGroup.from_config` builds the rig geometry
from the TOML.

The example intrinsics assume a 1024&times;512 sensor, but these frames are
960&times;480, so &mdash; exactly like `dev/run_pipeline_images.py` &mdash; we recenter the
principal point onto the real image center before calibration. We also read the
`[bundle_adjustment]` section, which says which parameters to hold *fixed* / *share*
during calibration (this anchors the world gauge).

In [ ]:
skeleton = Skeleton.fly()

with open(CONFIG, "rb") as f:
    cfg = tomllib.load(f)

probe = iio.imread(IMAGES_DIR / "camera_0_img_000000.jpg")
H, W = probe.shape[:2]
cfg.setdefault("camera_defaults", {})["principal_point_px"] = [(W - 1) / 2, (H - 1) / 2]
cameras = CameraGroup.from_config(cfg)

ba = cfg.get("bundle_adjustment", {})
calibrate_kwargs = {"fixed": ba.get("fixed", []), "shared": ba.get("shared", [])}

print(
    f"skeleton: {skeleton.name!r}  {skeleton.n_points} points, "
    f"{len(skeleton.bones)} bones, {skeleton.n_limbs} limbs"
)
print(f"limbs: {skeleton.limb_names}")
print(
    f"frame size (H, W) = {(H, W)};  principal point -> "
    f"{cfg['camera_defaults']['principal_point_px']}"
)
for c in cameras:
    print(f"  {c.name:>3}: center={np.round(c.position, 1)}  focal={c.intr[0]:.0f}px")

In [ ]:
from deeperfly.cameras import Camera


def plot_camera(camera: Camera, ax, length=None, **kwargs):
    "Draw a camera as an RGB axis triad at its world center"
    if length is None:
        length = np.linalg.norm(camera.tvec) * 0.2
    for axis, color in zip(camera.rmat, "rgb"):
        ax.plot(
            *zip(camera.position, camera.position + axis * length),
            color=color,
            **kwargs,
        )

In [ ]:
# The 7 cameras orbit the world origin (where the fly sits). Plot their centers.
fig = plt.figure(figsize=(6, 5))
ax = fig.add_subplot(projection="3d")
for c in cameras:
    plot_camera(c, ax, length=10, alpha=0.8)
    ax.text(
        *c.position + np.array([0, 0, 10]),
        s=c.name,
        color="k",
        fontsize=9,
        ha="center",
        va="center",
    )
ax.scatter([0], [0], [0], color="k", marker="x", s=70)
ax.set_title("Camera rig: centers orbit the world origin")
ax.set_xlabel("x")
ax.set_ylabel("y")
ax.set_zlabel("z")
ax.set_aspect("equal")
ax.view_init(elev=30, azim=-60)
ax.set_zticks([0])
plt.tight_layout()
plt.show()

## Step 2 &mdash; load the 2D detector

The detector is a stacked-hourglass network shipped in two interchangeable
backends. We load the default JAX/Equinox port from its cached checkpoint.

`inference.fly_camera_layout` returns, per camera, which body **side** the 19
detector channels populate and whether the image must be **mirror-flipped** so the
fly faces the trained orientation (left cameras are flipped).

In [ ]:
ckpt = jax_weights_path() if BACKEND == "jax" else torch_weights_path()
model = backends.load_detector(BACKEND, ckpt)
sides, flips = inference.fly_camera_layout(CAMERA_NAMES)

print(f"loaded {BACKEND} detector from {ckpt}")
for n, s, fl in zip(CAMERA_NAMES, sides, flips):
    print(f"  {n:>3}: side={s:<5} flip={fl}")

## Step 3 &mdash; load synchronized frames

`deeperfly run` reads one video (or image folder) per camera. This recording is a
flat directory, so we assemble the frames into a `(V, T, H, W)` array: for frame
`t`, the 7 entries `frames[:, t]` are the synchronized views.

In [ ]:
n_views = len(CAMERA_NAMES)
frame_ids = list(range(N_FRAMES))
frames = np.empty((n_views, N_FRAMES, H, W), dtype=np.uint8)
for v in range(n_views):
    for ti, fid in enumerate(frame_ids):
        frames[v, ti] = iio.imread(IMAGES_DIR / f"camera_{v}_img_{fid:06d}.jpg")
print("frames:", frames.shape, frames.dtype)

fig, axes = plt.subplots(2, 4, figsize=(12, 4))
for v, ax in enumerate(axes.ravel()):
    if v < n_views:
        ax.imshow(frames[v, 0], cmap="gray")
        ax.set_title(f"camera_{v} = {CAMERA_NAMES[v]} ({sides[v]})", fontsize=8)
    ax.axis("off")
fig.suptitle("Synchronized raw frame 0 across the 7 cameras")
plt.tight_layout()
plt.show()

## Step 4 &mdash; preprocessing

`inference.preprocess` mirror-flips far-side cameras, resizes to the network input
size (256&times;512), and subtracts the training mean. Below: a right camera (`rm`,
un-flipped) and a left camera (`lm`, mirror-flipped) before and after.

In [ ]:
def show_preproc(v, ax_raw, ax_pre):
    raw = frames[v, 0]
    pre = np.asarray(
        inference.preprocess(raw, flip=flips[v])
    )  # (3, 256, 512), mean-subtracted
    disp = np.clip(pre[0] + inference.MEAN, 0, 1)  # undo mean for display
    ax_raw.imshow(raw, cmap="gray")
    ax_raw.set_title(f"{CAMERA_NAMES[v]} raw {raw.shape}", fontsize=8)
    ax_raw.axis("off")
    ax_pre.imshow(disp, cmap="gray")
    ax_pre.set_title(
        f"preprocessed {tuple(pre.shape[1:])}  flip={flips[v]}", fontsize=8
    )
    ax_pre.axis("off")


fig, axes = plt.subplots(2, 2, figsize=(9, 5))
show_preproc(1, axes[0, 0], axes[0, 1])  # rm: right side, not flipped
show_preproc(5, axes[1, 0], axes[1, 1])  # lm: left side, mirror-flipped
fig.suptitle("preprocess(): mirror far-side cameras, resize to 256x512, subtract mean")
plt.tight_layout()
plt.show()

## Step 5 &mdash; heatmaps

`backends.predict_heatmaps` runs the forward pass, returning one heatmap per joint
(19 single-side channels) at the network's output resolution. The peak of each
heatmap is the predicted joint location. We overlay a few onto the input for one
right-side camera.

In [ ]:
inputs = np.stack(
    [
        np.asarray(inference.preprocess(frames[v, 0], flip=flips[v]))
        for v in range(n_views)
    ]
)
heatmaps = backends.predict_heatmaps(model, inputs)  # (V, J, Hh, Ww)
print("heatmaps:", heatmaps.shape, "  (views, joints, Hh, Ww)")

v = 1  # 'rm': a right-side, un-flipped camera -> channels 0..18 are right-side joints
hm = np.asarray(heatmaps[v])
disp = np.clip(
    np.asarray(inference.preprocess(frames[v, 0], flip=flips[v]))[0] + inference.MEAN,
    0,
    1,
)
ext = [0, disp.shape[1], disp.shape[0], 0]  # stretch heatmap onto the 256x512 input
J = hm.shape[0]

fig, axes = plt.subplots(2, 4, figsize=(13, 5))
axes = axes.ravel()
axes[0].imshow(disp, cmap="gray")
axes[0].imshow(hm.max(0), cmap="hot", alpha=0.55, extent=ext)
axes[0].set_title(f"{CAMERA_NAMES[v]}: max over {J} joints", fontsize=8)
axes[0].axis("off")
for i in range(min(7, J)):
    ax = axes[i + 1]
    ax.imshow(disp, cmap="gray")
    ax.imshow(hm[i], cmap="hot", alpha=0.6, extent=ext)
    ax.set_title(skeleton.joint_names[i], fontsize=7)
    ax.axis("off")
fig.suptitle("Stacked-hourglass heatmaps (one per tracked joint)")
plt.tight_layout()
plt.show()

## Step 6 &mdash; decode peaks and assemble the skeleton

`heatmap_to_points` takes the arg-max of each heatmap (a normalized `(x, y)` plus a
confidence); `assemble_skeleton` scatters each camera's 19 single-side detections
into the full 38-point skeleton (right cameras &rarr; `0..18`, mirror-flipped left
cameras &rarr; `19..37` with the flip undone) and scales back to original pixels.

In [ ]:
points_norm, conf0 = inference.heatmap_to_points(heatmaps)  # (V, J, 2) in [0,1], (V, J)
image_size = [(W, H)] * n_views  # (width, height) per camera
pts0, c0 = inference.assemble_skeleton(
    np.asarray(points_norm), np.asarray(conf0), sides, flips, image_size
)
print("assembled 2D points:", pts0.shape, " confidence:", c0.shape)

fig = viz.overlay_grid(
    pts0,
    skeleton,
    images=[frames[v, 0] for v in range(n_views)],
    camera_names=CAMERA_NAMES,
    ncols=4,
)
fig.suptitle("Decoded 2D skeleton overlaid on frame 0 (joint opacity = confidence)")
plt.show()

## Step 7 &mdash; detect the whole sequence

`inference.detect_sequence` repeats steps 4&ndash;6 for every frame, giving the full
2D result: `pts2d` of shape `(V, T, 38, 2)` in pixels and `conf` of shape
`(V, T, 38)`. This is the array `deeperfly run` feeds into the geometry pipeline.

In [ ]:
pts2d, conf = inference.detect_sequence(model, frames, sides, flips)
print("pts2d:", pts2d.shape, "  conf:", conf.shape)

fig, (axa, axb) = plt.subplots(1, 2, figsize=(13, 4))
im = axa.imshow(np.nanmean(conf, axis=1), aspect="auto", cmap="viridis", vmin=0, vmax=1)
axa.set_yticks(range(n_views))
axa.set_yticklabels(CAMERA_NAMES)
axa.set_xlabel("joint index  (0..18 right, 19..37 left)")
axa.set_title("Mean detector confidence per (camera, joint)")
fig.colorbar(im, ax=axa, fraction=0.046)

j, v = 4, 1  # RF tarsus tip seen by camera rm
axb.plot(pts2d[v, :, j, 0], label="x")
axb.plot(pts2d[v, :, j, 1], label="y")
axb.set_xlabel("frame")
axb.set_ylabel("pixel")
axb.legend()
axb.set_title(f"{CAMERA_NAMES[v]}: 2D track of {skeleton.joint_names[j]}")
plt.tight_layout()
plt.show()

## Step 8 &mdash; visibility masking

`run_from_points2d` first applies the skeleton's per-camera visibility: each camera
only sees one body side (plus shared midline points), so observations a camera
*cannot* see are set to NaN and never reach triangulation.

In [ ]:
pts2d_vis = apply_visibility(pts2d, skeleton, CAMERA_NAMES)
mask = skeleton.visibility_mask(CAMERA_NAMES)
before = int(np.isfinite(pts2d).all(-1).sum())
after = int(np.isfinite(pts2d_vis).all(-1).sum())
print(f"finite observations: {before} -> {after}  (masked {before - after} unviewable)")

fig, ax = plt.subplots(figsize=(8, 3))
ax.imshow(mask, aspect="auto", cmap="Greys_r", interpolation="nearest")
ax.set_yticks(range(n_views))
ax.set_yticklabels(CAMERA_NAMES)
ax.set_xlabel("joint index")
ax.set_title("Visibility mask (white = camera sees this joint)")
plt.tight_layout()
plt.show()

## Step 9 &mdash; calibration (bundle adjustment)

`pipeline.calibrate` treats the **fly itself** as the calibration target: it
flattens the frames into one point cloud and refines the cameras by bundle
adjustment, using detector confidences as per-observation weights, a robust loss,
and a soft bone-length prior. We compare reprojection error before vs after.

In [ ]:
# Reprojection error with the nominal (un-calibrated) cameras ...
pts3d_init = triangulate(cameras, pts2d_vis)
err_init = reprojection_error(cameras, pts3d_init, pts2d_vis)

# ... then refine the rig by fly-as-target bundle adjustment.
cameras_cal, ba_res = calibrate(cameras, pts2d_vis, conf, skeleton, **calibrate_kwargs)
pts3d_cal = triangulate(cameras_cal, pts2d_vis)
err_cal = reprojection_error(cameras_cal, pts3d_cal, pts2d_vis)

fi, fc = np.isfinite(err_init), np.isfinite(err_cal)
print(f"bundle adjustment: {ba_res.nfev} fn evals, final cost {ba_res.cost:.4g}")
print(
    f"median reproj error  before {np.median(err_init[fi]):.2f}px  ->  "
    f"after {np.median(err_cal[fc]):.2f}px"
)

fig, (axa, axb) = plt.subplots(1, 2, figsize=(13, 4))
bins = np.linspace(0, 40, 60)
axa.hist(
    err_init[fi],
    bins=bins,
    alpha=0.6,
    label=f"before (med {np.median(err_init[fi]):.1f})",
)
axa.hist(
    err_cal[fc], bins=bins, alpha=0.6, label=f"after (med {np.median(err_cal[fc]):.1f})"
)
axa.set_xlabel("reprojection error (px)")
axa.set_ylabel("count")
axa.legend()
axa.set_title("Reprojection error: nominal vs calibrated cameras")

shift = np.linalg.norm(
    np.array([c.position for c in cameras_cal])
    - np.array([c.position for c in cameras]),
    axis=1,
)
axb.bar(CAMERA_NAMES, shift, color="tab:purple")
axb.set_ylabel("camera-center shift (world units)")
axb.set_title("How far bundle adjustment moved each camera")
plt.tight_layout()
plt.show()

## Step 10 &mdash; triangulation with outlier rejection

`pipeline.reconstruct` triangulates each frame, then greedily drops the single
worst-reprojecting view of any still-offending point and re-triangulates &mdash;
removing gross 2D outliers without discarding the good views that they corrupted.
The result is the 3D pose sequence `(T, 38, 3)`.

In [ ]:
pts3d, pts2d_clean, reproj = reconstruct(
    cameras_cal, pts2d_vis, reproj_threshold=40.0, max_drops=5
)
fin = np.isfinite(reproj)
n_tri = int(np.isfinite(pts3d).all(-1).sum())
print(f"3D points: {pts3d.shape};  triangulated {n_tri} of {pts3d[..., 0].size}")
print(
    f"reproj error (px): median {np.median(reproj[fin]):.2f}  "
    f"p95 {np.percentile(reproj[fin], 95):.2f}"
)

t = N_FRAMES // 2
fig = plt.figure(figsize=(11, 5))
for k, (elev, azim) in enumerate([(20, -60), (85, -90)]):
    ax = fig.add_subplot(1, 2, k + 1, projection="3d")
    viz.plot_skeleton_3d(pts3d[t], skeleton, ax=ax, elev=elev, azim=azim)
    ax.set_title(f"frame {t}  (elev={elev}, azim={azim})", fontsize=9)
fig.suptitle("Triangulated 3D pose after reprojection-outlier rejection")
plt.tight_layout()
plt.show()

## Step 11 &mdash; temporal smoothing

The last stage applies a NaN-aware 1-Euro filter along time, suppressing jitter
while staying responsive to fast motion.

In [ ]:
pts3d_sm = smooth_one_euro(pts3d, FPS)

j = 4  # RF tarsus tip
fig, axes = plt.subplots(3, 1, figsize=(10, 6), sharex=True)
for k, lab in enumerate("xyz"):
    axes[k].plot(pts3d[:, j, k], ".-", alpha=0.5, label="raw")
    axes[k].plot(pts3d_sm[:, j, k], "-", lw=2, label="1-Euro")
    axes[k].set_ylabel(lab)
axes[0].legend(loc="upper right")
axes[0].set_title(f"Temporal smoothing of {skeleton.joint_names[j]} (3D position)")
axes[-1].set_xlabel("frame")
plt.tight_layout()
plt.show()

## Step 12 &mdash; the same thing in one call, and save the result

Everything from Step 8 on &mdash; visibility &rarr; calibrate &rarr; reconstruct &rarr;
smooth &mdash; is exactly what `pipeline.run_from_points2d` (and therefore
`deeperfly run`) does internally. We call it directly on the raw `pts2d` / `conf`,
save the `PoseResult` to HDF5, and reload it to confirm the round-trip.

In [ ]:
result = run_from_points2d(
    cameras,
    skeleton,
    pts2d,
    conf,
    do_calibrate=True,
    calibrate_kwargs=calibrate_kwargs,
    correct="reproject",
    smooth="one_euro",
    fps=FPS,
    meta={"source": str(IMAGES_DIR), "backend": BACKEND, "n_frames_input": N_FRAMES},
)

out = REPO / "results" / "fly_pose_walkthrough.h5"
out.parent.mkdir(parents=True, exist_ok=True)
result.save(out)

re = result.reproj_error
fr = np.isfinite(re)
print(f"wrote {out}")
print(f"  views x frames x points : {result.pts2d.shape}")
print(f"  3D points               : {result.pts3d.shape}")
print(
    f"  reprojection error (px) : median {np.median(re[fr]):.2f}  "
    f"p95 {np.percentile(re[fr], 95):.2f}"
)

reloaded = PoseResult.load(out)
print(
    f"  reloaded {reloaded.n_views} views x {reloaded.n_frames} frames; "
    f"has 3D = {reloaded.pts3d is not None}"
)

### Sanity check: reproject the calibrated 3D back onto every camera

If calibration and triangulation are consistent, projecting the recovered 3D pose
through the calibrated cameras should land on the fly in each raw view.

In [ ]:
t = 0
proj = result.cameras.project(result.pts3d[t])  # (V, 38, 2)
fig = viz.overlay_grid(
    proj,
    skeleton,
    images=[frames[v, t] for v in range(n_views)],
    camera_names=CAMERA_NAMES,
    ncols=4,
)
fig.suptitle("Calibrated 3D pose reprojected onto every camera (frame 0)")
plt.show()

## Mapping back to the CLI

| Notebook step | CLI / library call |
| --- | --- |
| Steps 1&ndash;2 | `CameraGroup.from_config`, `Skeleton.fly()`, `backends.load_detector` |
| Steps 3&ndash;7 | `inference.fly_camera_layout` + `inference.detect_sequence` |
| Steps 8&ndash;12 | `pipeline.run_from_points2d` (visibility &rarr; calibrate &rarr; reconstruct &rarr; smooth &rarr; save) |

The equivalent one-liner (note: the bare `deeperfly run` CLI expects one
video/folder *per camera name* and calls `calibrate` with its defaults; this flat
recording and the principal-point / gauge settings match `dev/run_pipeline_images.py`):

```bash
uv run python dev/run_pipeline_images.py --images data/images --frames 60 \
    --out results/fly_pose_walkthrough.h5

# render the result:
uv run --extra viz deeperfly visualize --in results/fly_pose_walkthrough.h5 \
    --out fly_pose3d.mp4 --mode 3d
```